In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import cross_val_predict

In [2]:
#Datasetin lataus ja rajaus
df_raw = pd.read_csv("cicids2017_cleaned.csv")
df100k = df_raw.sample(n=100000, random_state=42)
df = df100k[[
'Init_Win_bytes_forward',
'Flow IAT Mean',
'Packet Length Std',
'Subflow Fwd Bytes',
'Flow Duration',
'Bwd Packet Length Mean',
'Total Length of Fwd Packets',
'PSH Flag Count',
'Flow Packets/s',
'Destination Port',
'Attack Type'
]]

X = df.drop('Attack Type', axis=1)
y = df['Attack Type']

In [3]:
# Splittaus
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = ImbPipeline([ #HUOM TEHDÄÄN TÄMÄ SMOTELLA, MUUTEN EI TUNNISTA WEB ATTACKS LUOKKAA
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('smote', SMOTE(random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42))
])

param_grid = {
    'pca__n_components': [5, 8, 0.95],
    'dt__max_depth': [5, 10], #Kuinka syvä puusta tulee. Overfittaantuu herkästi, siksi ei isoja lukuja.
    'dt__criterion': ['gini', 'entropy'], #Datan jakomalli
    'dt__min_samples_split': [5, 10], #Paljonko dataa vaaditaan, solmu jaetaan kahtia? Overfittaantuu herkästi, siksi ei pienempiä lukuja.
    'dt__min_samples_leaf': [2, 5, 10] #Kuinka monta datapistettä voi olla yhdessä lopullisessa mallissa vähintään
}   

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=4,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1 #HUOM, MAHDOLLISTAA KAIKKIEN YTIMIEN KÄYTÖN, JA SITEN NOPEAMMAN TESTIN
)

grid_search.fit(X_train_full, y_train_full)

print(f"Parhaat parametrit: {grid_search.best_params_}")
print("_________________________________________________________")

best_model = grid_search.best_estimator_


# TRAINING
train_preds = best_model.predict(X_train_full)
print("\n=== Training-raportti (Train Set) ===")
print(classification_report(y_train_full, train_preds))
print(confusion_matrix(y_train_full, train_preds))
print("_________________________________________________________")


# VALIDOINTI
val_preds = cross_val_predict(best_model, X_train_full, y_train_full, cv=5)
print("\n=== Validointiraportti (Cross-Validation Predictions) ===")
print(classification_report(y_train_full, val_preds))
print(confusion_matrix(y_train_full, val_preds))
print("_________________________________________________________")


# TESTI
test_preds = best_model.predict(X_test)
print("\n=== Lopullinen testi (Test Set): ===")
print(classification_report(y_test, test_preds))
print(confusion_matrix(y_test, test_preds))

Fitting 4 folds for each of 72 candidates, totalling 288 fits
Parhaat parametrit: {'dt__criterion': 'entropy', 'dt__max_depth': 10, 'dt__min_samples_leaf': 2, 'dt__min_samples_split': 10, 'pca__n_components': 0.95}
_________________________________________________________

=== Training-raportti (Train Set) ===
                precision    recall  f1-score   support

          Bots       0.07      1.00      0.14        78
   Brute Force       0.70      0.99      0.82       310
          DDoS       0.89      0.98      0.93      4106
           DoS       0.88      0.95      0.91      6174
Normal Traffic       1.00      0.94      0.97     66378
 Port Scanning       0.71      1.00      0.83      2879
   Web Attacks       0.13      1.00      0.23        75

      accuracy                           0.95     80000
     macro avg       0.63      0.98      0.69     80000
  weighted avg       0.97      0.95      0.96     80000

[[   78     0     0     0     0     0     0]
 [    0   307     0     